This notebook generates Cells with cloelib given a resolution at which to interpolate the redshift distributions

In [23]:
# General imports
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate
from copy import deepcopy
import os
import time
import astropy.io.fits as fits
from scipy import interpolate
from scipy import integrate
from scipy.ndimage import gaussian_filter1d

# Euclid imports
from cloelib.cosmology.camb_cosmology import CAMBBackground, CAMBLinearPerturbations, CAMBNonLinearPerturbations
from cloelib.cosmology.HMcode2020Emu_cosmology import HMemuLinearPerturbations, HMemuNonLinearPerturbations
from cloelib.observables.photo import ShearTracer, PositionsTracer
from cloelib.summary_statistics.angular_two_point import AngularTwoPoint
import euclidlib as el
from cloelike.EuclidLikelihood_photo_Cls import EuclidLikelihood_WL

import heracles

import seaborn as sns
sns.set_theme(style='ticks')
sns.set_palette(sns.color_palette("Paired"))
#plt.style.use('/Users/jcoloma/phd/ice/euclid/beautiful_plots.mplstyle')

## 0. Set up

In [24]:
# Choose a resolution to interpolate the n(z)'s
resolution = 256 # 1000 # 2000

# path adnd file name where to save cells
path_out =  '/Users/jcoloma/phd/ice/euclid/cosmosis_vs_cloe/draft/files/cloelib/'
filename = 'cells_cloelib_' + str(resolution) + '.fits'

## 1. Background

In [25]:
background = CAMBBackground(H0 = 67., 
                            Omega_cdm0 = 0.2686, 
                            Omega_b0 = 0.049,
                            w0 = -1, 
                            wa = 0, 
                            Omega_k0 = 0.0, 
                            ns = 0.96, 
                            As = 2.1e-9,
                            mnu = 0.06,
                            N_mnu = 1,                          
                            gamma_MG=0,
                            )

zs = np.linspace(1e-6, 3., 256) 
kmin = 5e-4
kmax = 50
nk = 400
ks = np.logspace(np.log10(kmin), np.log10(kmax), nk)

linear_perturbations = CAMBLinearPerturbations(background, zs)
nonlinear_perturbations = CAMBNonLinearPerturbations(background, zs, 'mead2020')
#linear_perturbations = HMemuLinearPerturbations(background, zs)
#nonlinear_perturbations = HMemuNonLinearPerturbations(background, linear_perturbations, zs)

# print(background.interface_args["CAMBparams"])


Note: redshifts have been re-sorted (earliest first)
Note: redshifts have been re-sorted (earliest first)


## 2. Redshift distributions

In [26]:
# Read
path_in = '/Users/jcoloma/phd/ice/euclid/cosmosis_vs_cloe/draft/files/nz/'
nz_file_she = np.loadtxt(path_in + 'nz_she_fs2_vis24_25559.txt')

z_she = nz_file_she[:,0]
nz_she = nz_file_she[:,1:].T

nz_file_pos = np.loadtxt(path_in + 'nz_pos_fs2_h22_24607.txt')
z_pos = nz_file_pos[:,0]
nz_pos = nz_file_pos[:,1:].T

# Normalize
def normalize_dndz(nz, z):
    nz_norm = np.zeros(nz.shape)
    for i in range(nz.shape[0]):
        nz_norm[i] = nz[i] / integrate.trapezoid(nz[i], z)
    return nz_norm

nz_she_norm =  normalize_dndz(nz_she, z_she)
nz_pos_norm =  normalize_dndz(nz_pos, z_pos)

# Interpolate
N_interp = resolution
z_interp = np.linspace(1e-6, 3.0, N_interp)  #Full redshift range 
#z_interp = np.linspace(max(z_she[0], z_pos[0]), min(z_she[-1], z_pos[-1]), N_interp) # Overlapping range

dndz_she = np.zeros((6, len(z_interp)))
for i in range(6):
    dndz_she[i,:] = np.interp(z_interp, z_she, nz_she_norm[i,:])

dndz_pos = np.zeros((6, len(z_interp)))
for i in range(6):
    dndz_pos[i,:] = np.interp(z_interp, z_pos, nz_pos_norm[i,:])

In [27]:
# CosmoSIS function to get C_IA
def compute_c1_baseline():
    C1_M_sun = 5e-14  # h^-2 M_S^-1 Mpc^3
    M_sun = 1.9891e30  # kg
    Mpc_in_m = 3.0857e22  # meters
    C1_SI = C1_M_sun / M_sun * (Mpc_in_m) ** 3  # h^-2 kg^-1 m^3
    # rho_crit_0 = 3 H^2 / 8 pi G
    G = 6.67384e-11  # m^3 kg^-1 s^-2
    H = 100  # ?| h km s^-1 Mpc^-1
    H_SI = H * 1000.0 / Mpc_in_m  # h s^-1
    rho_crit_0 = 3 * H_SI ** 2 / (8 * np.pi * G)  #  h^2 kg m^-3
    f = C1_SI * rho_crit_0
    return f

cia = compute_c1_baseline()

## 3. Tracers

In [28]:
tracer_she = ShearTracer(perturbations=nonlinear_perturbations, 
                            dndz=dndz_she,
                            z=z_interp,
                            nuisance_params={ 'AIA': 0., 'CIA': 0., 'EtaIA':0., #'AIA': 0.16, 'CIA': cia, 'EtaIA':1.66,
                                              'multiplicative_bias_1': 0.0, 'multiplicative_bias_2': 0.0,
                                              'multiplicative_bias_3': 0.0, 'multiplicative_bias_4': 0.0,
                                              'multiplicative_bias_5': 0.0, 'multiplicative_bias_6': 0.0,
                                              'dz_shear_1': 0.0, 'dz_shear_2': 0.0,
                                              'dz_shear_3': 0.0, 'dz_shear_4': 0.0,
                                              'dz_shear_5': 0.0, 'dz_shear_6': 0.0}
                              )  

tracer_pos = PositionsTracer(perturbations=nonlinear_perturbations, 
                            dndz=dndz_pos,
                            z=z_interp,
                            galaxy_bias_model='per_bin',
                            nuisance_params={ 'b1_photo_bin0': 1.14, 'b1_photo_bin1': 1.25, 
                                              'b1_photo_bin2': 1.40, 'b1_photo_bin3': 1.45,
                                              'b1_photo_bin4': 1.60, 'b1_photo_bin5': 1.75, 
                                              'dz_pos_1': 0.0, 'dz_pos_2': 0.0,
                                              'dz_pos_3': 0.0, 'dz_pos_4': 0.0,
                                              'dz_pos_5': 0.0, 'dz_pos_6': 0.0,
                                              'magnification_bias_1': 0., 'magnification_bias_2': 0.,
                                              'magnification_bias_3': 0., 'magnification_bias_4': 0.,
                                              'magnification_bias_5': 0., 'magnification_bias_6': 0.,}
                                              #'magnification_bias_1': 2*(0.52-1), 'magnification_bias_2': 2*(0.68-1),
                                              #'magnification_bias_3': 2*(0.65-1), 'magnification_bias_4': 2*(0.80-1),
                                              #'magnification_bias_5': 2*(1.18-1), 'magnification_bias_6': 2*(1.97-1),}
                                            
                            )

## 4. Compute Cells

In [29]:
nl = 32
ells= np.logspace(np.log10(10), np.log10(3000), nl)

twopoint_sheshe = AngularTwoPoint(tracer_she, tracer_she)
twopoint_pospos = AngularTwoPoint(tracer_pos, tracer_pos)
twopoint_posshe = AngularTwoPoint(tracer_pos, tracer_she)

# Decoupled Cls
cells = {**twopoint_sheshe.get_Cl(ells, nl, ks),
         **twopoint_pospos.get_Cl(ells, nl, ks),
         **twopoint_posshe.get_Cl(ells, nl, ks)}

In [30]:
# Save to file
heracles.write(path_out + filename, cells, clobber=True)